# 05 ML GNN Embeddings — Group 1: Original (Not Fine-Tuned)

**Group 1.** Trains regressors on the original GNN embeddings produced by the default link-prediction training objective — no reconstruction loss, no temporal warm-starting.

| Dataset | Model | Dim | Target |
|---|---|---|---|
| `graphsage_srisk_dataset.parquet` | GraphSAGE v1 | 64 | `log_systemic_risk_label` |
| `node2vec_srisk_dataset.parquet` | Node2Vec v1 | 64 | `log_systemic_risk_label` |

In [ ]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

sys.path.insert(0, os.path.abspath('../..'))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]

In [2]:
print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Dataset

In [3]:
df, feature_cols = load_gnn_dataset(PROJECT_ROOT, filename="graphsage_srisk_dataset.parquet")
print(df.shape, len(feature_cols))

df_1, feature_cols_1 = load_gnn_dataset(PROJECT_ROOT, target_col="log_systemic_risk_label", filename="node2vec_srisk_dataset.parquet")
print(df_1.shape, len(feature_cols_1))

(145536, 69) 64
(145536, 69) 64


In [4]:
trainer = ModelTrainer(
    df=df,
    feature_cols=feature_cols,
    target_col="log_systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

In [5]:
trainer_1 = ModelTrainer(
    df=df_1,
    feature_cols=feature_cols_1,
    target_col="log_systemic_risk_label",
)

trainer_1.train_df.shape, trainer_1.val_df.shape, trainer_1.test_df.shape

((109152, 69), (18192, 69), (13644, 69))

## Define Models

In [ ]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]
TOP1_COLS    = ["model", "train_top1_mae", "validation_top1_mae", "train_top1_rmse", "validation_top1_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

## Train And Store

In [ ]:
trainer.train_all(candidate_models)
trainer.leaderboard()[DISPLAY_COLS]

In [ ]:
trainer.leaderboard()[TOP1_COLS]

In [ ]:
trainer_1.train_all(candidate_models)
trainer_1.leaderboard()[DISPLAY_COLS]

In [ ]:
trainer_1.leaderboard()[TOP1_COLS]

## Hyperparameter Tuning

In [ ]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model, param_distributions,
        n_iter=n_iter, cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42, n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    return search.best_params_

RF_PARAMS = {
    "model__n_estimators":     randint(100, 600),
    "model__max_depth":        [None, 5, 10, 15, 20, 30],
    "model__min_samples_leaf": randint(1, 20),
    "model__min_samples_split":randint(2, 20),
    "model__max_features":     ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          randint(100, 600),
    "model__max_depth":         [3, 4, 5, 6, 8, None],
    "model__learning_rate":     loguniform(0.005, 0.3),
    "model__min_samples_leaf":  randint(5, 100),
    "model__l2_regularization": loguniform(1e-4, 1.0),
    "model__max_leaf_nodes":    randint(15, 60),
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     randint(100, 800),
    "model__max_depth":        randint(3, 10),
    "model__learning_rate":    loguniform(0.005, 0.3),
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": randint(1, 10),
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        loguniform(1e-5, 1.0),
    "model__reg_lambda":       loguniform(1e-5, 2.0),
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":               loguniform(1e-5, 0.1),
    "model__learning_rate_init":  loguniform(1e-4, 0.05),
    "model__learning_rate":       ["constant", "adaptive"],
    "model__batch_size":          [32, 64, 128, "auto"],
}

In [ ]:
# GraphSAGE v1 tuning
tune(trainer, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  "Random Forest (tuned)")
tune(trainer, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  "Gradient Boosting (tuned)")
tune(trainer, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, "XGBoost (tuned)")
tune(trainer, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, "MLP (tuned)")
trainer.leaderboard()[DISPLAY_COLS]

In [ ]:
# Node2Vec v1 tuning
tune(trainer_1, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  "Random Forest (tuned)")
tune(trainer_1, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  "Gradient Boosting (tuned)")
tune(trainer_1, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, "XGBoost (tuned)")
tune(trainer_1, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, "MLP (tuned)")
trainer_1.leaderboard()[DISPLAY_COLS]